# Le consegne in ritardo: la regressione logistica

*Metodi Matematici e Statistici per le Decisioni Economiche · A.A. 2026/27 · Lezione 3*

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincnardelli/msd/blob/main/codice/3/regressione_shipping.ipynb)

Un'azienda di e-commerce vuole capire quali spedizioni arrivano in ritardo. `shipping.csv`, 10.999 spedizioni: `reached_time` (**1** in ritardo, **0** puntuale: è già 0/1 nel file), `mode_of_shipment` (`Flight`, `Ship`, `Road`), `customer_rating` (da 1 a 5), `cost_of_the_product` (dollari), `prior_purchases` (acquisti precedenti del cliente), `product_importance` (`low`, `medium`, `high`), `gender` (`F`, `M`), `discount_offered` (sconto, in punti percentuali), `weight_in_gms` (peso, in grammi).

In [1]:
import polars as pl                                 # tabelle di dati
import statsmodels.formula.api as smf               # modelli statistici
import math                                         # per il numero e
from sklearn.metrics import confusion_matrix, classification_report   # matrice di confusione e metriche

In [2]:
spedizioni = pl.read_csv("https://www.vincnardelli.com/msd/dati/shipping.csv")
spedizioni.head()

mode_of_shipment,customer_rating,cost_of_the_product,prior_purchases,product_importance,gender,discount_offered,weight_in_gms,reached_time
str,i64,i64,i64,str,str,i64,i64,i64
"""Flight""",2,177,3,"""low""","""F""",44,1233,1
"""Flight""",5,216,2,"""low""","""M""",59,3088,1
"""Flight""",2,183,4,"""low""","""M""",48,3374,1
"""Flight""",3,176,4,"""medium""","""M""",10,1177,1
"""Flight""",2,184,3,"""medium""","""F""",46,2484,1


### 1. Lo sconto fa arrivare in ritardo? Stima la regressione logistica di `reached_time` in funzione di `discount_offered`.

**Cosa mi chiede?** y = `reached_time`, x = `discount_offered`. Comando: `smf.logit`, e `summary2()` per la tabella: come `summary()`, ma mostra anche l'AIC.

In [3]:
modello = smf.logit("reached_time ~ discount_offered", data=spedizioni).fit()
modello.summary2()

Optimization terminated successfully.
         Current function value: 0.557125
         Iterations 8


<class 'statsmodels.iolib.summary2.Summary'>
"""
                         Results: Logit
=================================================================
Model:              Logit            Method:           MLE       
Dependent Variable: reached_time     Pseudo R-squared: 0.174     
Date:               2026-11-25 12:18 AIC:              12259.6399
No. Observations:   10999            BIC:              12274.2510
Df Model:           1                Log-Likelihood:   -6127.8   
Df Residuals:       10997            LL-Null:          -7417.0   
Converged:          1.0000           LLR p-value:      0.0000    
No. Iterations:     8.0000           Scale:            1.0000    
-----------------------------------------------------------------
                  Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-----------------------------------------------------------------
Intercept        -0.7365   0.0352 -20.9117 0.0000 -0.8055 -0.6675
discount_offered  0.1257   0.0043  29.0764 0.0000  0.1172  0.1342
=================================================================

"""

Coefficienti: intercetta -0.74 e `discount_offered` 0.1257, sono log-odds (positivo: più sconto, più probabile il ritardo)  
p-value 0.000 <= 0.05: lo sconto è significativo; Pseudo R-squared 0.17, AIC 12259.6.

### 2. Di quanto cambiano gli odds di ritardo per ogni punto di sconto in più? E per 10 punti?

**Cosa mi chiede?** L'odds ratio: `math.e` elevato al coefficiente.

In [4]:
math.e ** modello.params

Intercept           0.478783
discount_offered    1.133938
dtype: float64

`discount_offered` 1.134: ogni punto di sconto in più, gli odds di ritardo si moltiplicano per 1.134 (+13%).

In [5]:
math.e ** (10 * modello.params["discount_offered"])

np.float64(3.5147521701207656)

10 punti di sconto in più: odds x 3.51.

### 3. Con che probabilità arriva in ritardo una spedizione con sconto 5, 10 e 20? Quanto valgono log-odds e odds?

**Cosa mi chiede?** Tre comandi: i log-odds con `predict(..., which="linear")`, gli odds con `math.e` elevato ai log-odds, la probabilità con `predict`.

Le tre spedizioni.

In [6]:
nuovi = {"discount_offered": [5, 10, 20]}

In [7]:
modello.predict(nuovi, which="linear")

0   -0.108022
1    0.520462
2    1.777431
dtype: float64

Log-odds: -0.11, 0.52 e 1.78 (la retta: -0.74 + 0.1257 * sconto).

In [8]:
math.e ** modello.predict(nuovi, which="linear")

0    0.897607
1    1.682805
2    5.914643
dtype: float64

Odds: 0.90, 1.68 e 5.91  
con sconto 20: circa 5.9 spedizioni in ritardo per ogni spedizione puntuale.

In [9]:
modello.predict(nuovi)

0    0.473021
1    0.627256
2    0.855379
dtype: float64

Probabilità: 0.473, 0.627 e 0.855.

### 4. Metti nel modello tutte le variabili. Quali sono significative?

**Cosa mi chiede?** Una logistica multipla con tutte le x. Le variabili di testo (`mode_of_shipment`, `product_importance`, `gender`) vanno nella formula così come sono: le dummy le crea statsmodels. Comando: `summary2()`, che mostra anche l'AIC.

In [10]:
tutte = smf.logit("reached_time ~ mode_of_shipment + customer_rating + cost_of_the_product + prior_purchases + product_importance + gender + discount_offered + weight_in_gms", data=spedizioni).fit()
tutte.summary2()

Optimization terminated successfully.
         Current function value: 0.546995
         Iterations 8


<class 'statsmodels.iolib.summary2.Summary'>
"""
                               Results: Logit
=============================================================================
Model:                  Logit                Method:               MLE       
Dependent Variable:     reached_time         Pseudo R-squared:     0.189     
Date:                   2026-11-25 12:18     AIC:                  12054.7890
No. Observations:       10999                BIC:                  12135.1502
Df Model:               10                   Log-Likelihood:       -6016.4   
Df Residuals:           10988                LL-Null:              -7417.0   
Converged:              1.0000               LLR p-value:          0.0000    
No. Iterations:         8.0000               Scale:                1.0000    
-----------------------------------------------------------------------------
                              Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-----------------------------------------------------------------------------
Intercept                     1.2619   0.1911   6.6036 0.0000  0.8874  1.6364
mode_of_shipment[T.Road]     -0.0328   0.0766  -0.4279 0.6687 -0.1830  0.1174
mode_of_shipment[T.Ship]     -0.0150   0.0602  -0.2491 0.8033 -0.1329  0.1029
product_importance[T.low]    -0.3560   0.0835  -4.2615 0.0000 -0.5197 -0.1923
product_importance[T.medium] -0.3400   0.0838  -4.0572 0.0000 -0.5043 -0.1758
gender[T.M]                   0.0527   0.0436   1.2099 0.2263 -0.0327  0.1382
customer_rating               0.0243   0.0154   1.5740 0.1155 -0.0060  0.0546
cost_of_the_product          -0.0026   0.0005  -5.2505 0.0000 -0.0035 -0.0016
prior_purchases              -0.0822   0.0151  -5.4384 0.0000 -0.1119 -0.0526
discount_offered              0.1138   0.0044  25.6705 0.0000  0.1052  0.1225
weight_in_gms                -0.0002   0.0000 -14.0949 0.0000 -0.0002 -0.0002
=============================================================================

"""

Significative (p-value <= 0.05): `product_importance`, `cost_of_the_product`, `prior_purchases`, `discount_offered`, `weight_in_gms`  
non significative: `mode_of_shipment` (p 0.67 e 0.80), `gender` (p 0.23), `customer_rating` (p 0.12)  
`product_importance[T.low]` -0.36: rispetto a high (il riferimento), i prodotti poco importanti hanno odds di ritardo x 0.70.

### 5. Togliamo le variabili non significative: il modello migliora? Usa l'AIC.

**Cosa mi chiede?** Si sceglie il modello con l'AIC **più basso**.

In [11]:
ridotto = smf.logit("reached_time ~ product_importance + cost_of_the_product + prior_purchases + discount_offered + weight_in_gms", data=spedizioni).fit()
ridotto.summary2()

Optimization terminated successfully.
         Current function value: 0.547183
         Iterations 8


<class 'statsmodels.iolib.summary2.Summary'>
"""
                               Results: Logit
=============================================================================
Model:                  Logit                Method:               MLE       
Dependent Variable:     reached_time         Pseudo R-squared:     0.189     
Date:                   2026-11-25 12:18     AIC:                  12050.9416
No. Observations:       10999                BIC:                  12102.0805
Df Model:               6                    Log-Likelihood:       -6018.5   
Df Residuals:           10992                LL-Null:              -7417.0   
Converged:              1.0000               LLR p-value:          0.0000    
No. Iterations:         8.0000               Scale:                1.0000    
-----------------------------------------------------------------------------
                              Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
-----------------------------------------------------------------------------
Intercept                     1.3439   0.1784   7.5347 0.0000  0.9943  1.6935
product_importance[T.low]    -0.3573   0.0835  -4.2784 0.0000 -0.5210 -0.1936
product_importance[T.medium] -0.3420   0.0838  -4.0818 0.0000 -0.5062 -0.1778
cost_of_the_product          -0.0025   0.0005  -5.2192 0.0000 -0.0035 -0.0016
prior_purchases              -0.0823   0.0151  -5.4482 0.0000 -0.1120 -0.0527
discount_offered              0.1138   0.0044  25.6683 0.0000  0.1051  0.1225
weight_in_gms                -0.0002   0.0000 -14.0966 0.0000 -0.0002 -0.0002
=============================================================================

"""

Pseudo R-squared 0.189 (con tutte 0.189); AIC 12050.9 < 12054.8: meglio il modello ridotto.

In [12]:
modello.aic, tutte.aic, ridotto.aic

(np.float64(12259.63992489728),
 np.float64(12054.789014699221),
 np.float64(12050.94156812046))

AIC: solo sconto 12259.6, tutte 12054.8, ridotto 12050.9: il più basso è il migliore.

### 6. Quante spedizioni indovina il modello ridotto, con la soglia 0.5? Calcola la matrice di confusione.

**Cosa mi chiede?** Una colonna con la probabilità prevista, poi la realtà (`reached_time`) contro la previsione (`prob > 0.5`): `confusion_matrix`. Righe e colonne sono nello stesso ordine delle slide.

La probabilità di ritardo prevista per ogni spedizione.

In [13]:
spedizioni = spedizioni.with_columns(prob=ridotto.predict())

In [14]:
confusion_matrix(spedizioni["reached_time"], spedizioni["prob"] > 0.5)

array([[2558, 1878],
       [2085, 4478]])

Righe = reale, colonne = previsto, come nelle slide:

| | previsto 0 (puntuale) | previsto 1 (in ritardo) |
|---|---|---|
| **reale 0 (puntuale)** | 2558 veri negativi | 1878 falsi positivi |
| **reale 1 (in ritardo)** | 2085 falsi negativi | 4478 veri positivi |

### 7. Cosa succede con la soglia 0.3? E con 0.7?

**Cosa mi chiede?** La stessa matrice di confusione, cambiando solo la soglia.

In [15]:
confusion_matrix(spedizioni["reached_time"], spedizioni["prob"] > 0.3)

array([[ 159, 4277],
       [ 132, 6431]])

Soglia 0.3: [[159 4277] [132 6431]]: quasi tutte previste in ritardo, pochi falsi negativi (132) e tanti falsi positivi (4277).

In [16]:
confusion_matrix(spedizioni["reached_time"], spedizioni["prob"] > 0.7)

array([[4388,   48],
       [3837, 2726]])

Soglia 0.7: [[4388 48] [3837 2726]]: pochissimi falsi positivi (48), ma tanti falsi negativi (3837).

### 8. Con la soglia 0.5: quanto valgono accuratezza, sensitività e specificità?

**Cosa mi chiede?** Tutte le metriche in un output solo: `classification_report`, con gli stessi due argomenti della matrice di confusione.

Come si legge: **accuracy** è l'accuratezza. La **sensitività** è il **recall della classe positiva**, la riga `in ritardo (1)`; la **specificità** è il **recall della classe negativa**, la riga `puntuale (0)`: lo stesso calcolo, fatto per le due classi. **Precision** della riga `in ritardo (1)`: tra le spedizioni previste in ritardo, quante lo sono davvero.

In [17]:
print(classification_report(spedizioni["reached_time"], spedizioni["prob"] > 0.5, target_names=["puntuale (0)", "in ritardo (1)"], digits=3))

                precision    recall  f1-score   support

  puntuale (0)      0.551     0.577     0.563      4436
in ritardo (1)      0.705     0.682     0.693      6563

      accuracy                          0.640     10999
     macro avg      0.628     0.629     0.628     10999
  weighted avg      0.643     0.640     0.641     10999



Accuratezza 0.640; sensitività 0.682; specificità 0.577.

In [18]:
print(classification_report(spedizioni["reached_time"], spedizioni["prob"] > 0.7, target_names=["puntuale (0)", "in ritardo (1)"], digits=3))

                precision    recall  f1-score   support

  puntuale (0)      0.533     0.989     0.693      4436
in ritardo (1)      0.983     0.415     0.584      6563

      accuracy                          0.647     10999
     macro avg      0.758     0.702     0.639     10999
  weighted avg      0.802     0.647     0.628     10999



Con 0.7: sensitività 0.415, specificità 0.989: quasi nessun falso allarme, ma molti ritardi non visti.